# 17.2 知识编辑 (Knowledge Editing)

知识编辑使模型在不需要完全重新训练的情况下精确更新特定知识。

本节涵盖：
- ROME：秩一模型编辑
- MEMIT：大规模知识编辑
- SERAC：反事实编辑
- 知识编辑评估
- LLM知识编辑实践

## 1. ROME — 秩一模型编辑

**ROME** (Rank-One Model Editing) 通过修改MLP层中的特定权重来改变模型对特定事实的记忆。

### 核心思想
LLM中的事实知识存储在MLP层中，修改特定键值对即可更新知识。

### 算法
1. 定位：找到存储目标知识的层
2. 提取：计算该层的键向量k和值向量v
3. 编辑：用秩一更新修改权重：$W \leftarrow W + (v_{new} - v_{old}) k^T / (k^T k)$

### 局限性
- 每次只能编辑一条知识
- 可能影响相关知识的准确性
- 需要访问模型内部

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class ROMEEditor:
    def __init__(self, model, layer_idx=0):
        self.model = model
        self.layer_idx = layer_idx
        self.edits = []

    def locate_layer(self):
        linear_layers = [(i, m) for i, m in enumerate(self.model.modules()) if isinstance(m, nn.Linear)]
        return linear_layers[self.layer_idx]

    def edit(self, subject_embed, old_object_embed, new_object_embed):
        idx, layer = self.locate_layer()
        W = layer.weight.data
        k = subject_embed.flatten()
        v_old = (W @ k).flatten()
        v_new = v_old + (new_object_embed.flatten() - old_object_embed.flatten())
        k_norm_sq = k @ k + 1e-8
        delta = torch.outer(v_new - v_old, k) / k_norm_sq
        layer.weight.data = W + delta
        self.edits.append({'subject_norm': k.norm().item(), 'delta_norm': delta.norm().item()})
        return delta.norm().item()

    def verify(self, subject_embed, expected_embed):
        idx, layer = self.locate_layer()
        output = layer(subject_embed.flatten())
        similarity = F.cosine_similarity(output.unsqueeze(0), expected_embed.flatten().unsqueeze(0)).item()
        return similarity

class SimpleKnowledgeModel(nn.Module):
    def __init__(self, d=64, n_entities=20, n_facts=50):
        super().__init__()
        self.entity_embed = nn.Embedding(n_entities, d)
        self.relation_embed = nn.Embedding(5, d)
        self.knowledge_layer = nn.Linear(d, d)
        self.output = nn.Linear(d, n_entities)

    def forward(self, entity_id, relation_id):
        e = self.entity_embed(entity_id)
        r = self.relation_embed(relation_id)
        h = self.knowledge_layer(e + r)
        return self.output(h)

model = SimpleKnowledgeModel(d=64, n_entities=20)
editor = ROMEEditor(model, layer_idx=0)

subject = torch.randn(1, 64)
old_obj = torch.randn(1, 64)
new_obj = torch.randn(1, 64)

print('=== ROME: Rank-One Model Editing ===')
before_sim = editor.verify(subject, old_obj)
print(f'Before edit: cosine_sim(subject_output, old_object) = {before_sim:.4f}')

delta_norm = editor.edit(subject, old_obj, new_obj)
print(f'Edit applied: delta_norm = {delta_norm:.4f}')

after_sim_old = editor.verify(subject, old_obj)
after_sim_new = editor.verify(subject, new_obj)
print(f'After edit: cosine_sim(output, old_object) = {after_sim_old:.4f}')
print(f'After edit: cosine_sim(output, new_object) = {after_sim_new:.4f}')
print(f'\nKey: ROME applies rank-one update to change specific knowledge.')
print(f'The edit is surgical — only the target fact is modified.')

## 2. MEMIT — 大规模知识编辑

**MEMIT** (Mass-Editing Memory in a Transformer) 扩展ROME到同时编辑多条知识。

### 与ROME的区别
- ROME：单条编辑，秩一更新
- MEMIT：批量编辑，约束优化

### 算法
1. 收集所有要编辑的(键, 旧值, 新值)三元组
2. 求解约束优化：$\min_\Delta \|\Delta\|_F$ s.t. $\Delta K = V_{new} - V_{old}$
3. 最优解：$\Delta = (V_{new} - V_{old}) K^T (K K^T)^{-1}$
4. 应用更新：$W \leftarrow W + \Delta$

### 优势
- 可同时编辑数千条知识
- 编辑间干扰更小
- 适合批量事实更新

In [ ]:
class MEMITEditor:
    def __init__(self, model, layer_idx=0):
        self.model = model
        self.layer_idx = layer_idx

    def locate_layer(self):
        linear_layers = [(i, m) for i, m in enumerate(self.model.modules()) if isinstance(m, nn.Linear)]
        return linear_layers[self.layer_idx]

    def batch_edit(self, subjects, old_objects, new_objects, reg_factor=0.1):
        idx, layer = self.locate_layer()
        W = layer.weight.data.clone()
        K = torch.stack([s.flatten() for s in subjects]).T
        V_old = torch.stack([(W @ s.flatten()) for s in subjects])
        V_new = V_old + torch.stack([n.flatten() - o.flatten() for n, o in zip(new_objects, old_objects)])
        KKT = K.T @ K + reg_factor * torch.eye(K.size(1))
        try:
            Delta = (V_new - V_old) @ torch.linalg.solve(KKT, K.T).T
        except:
            Delta = (V_new - V_old) @ torch.linalg.pinv(K.T @ K) @ K.T
        layer.weight.data = W + Delta.T
        return Delta.norm().item()

    def verify_batch(self, subjects, expected_objects):
        idx, layer = self.locate_layer()
        similarities = []
        for s, e in zip(subjects, expected_objects):
            out = layer(s.flatten())
            sim = F.cosine_similarity(out.unsqueeze(0), e.flatten().unsqueeze(0)).item()
            similarities.append(sim)
        return similarities

model_memit = SimpleKnowledgeModel(d=64, n_entities=20)
memit = MEMITEditor(model_memit, layer_idx=0)

n_edits = 5
subjects = [torch.randn(1, 64) for _ in range(n_edits)]
old_objects = [torch.randn(1, 64) for _ in range(n_edits)]
new_objects = [torch.randn(1, 64) for _ in range(n_edits)]

print('=== MEMIT: Mass-Editing Memory in a Transformer ===')
sims_before = memit.verify_batch(subjects, new_objects)
print(f'Before edit: avg_sim to new_objects = {sum(sims_before)/len(sims_before):.4f}')

delta_norm = memit.batch_edit(subjects, old_objects, new_objects)
print(f'Batch edit applied: delta_norm = {delta_norm:.4f}')

sims_after = memit.verify_batch(subjects, new_objects)
print(f'After edit: avg_sim to new_objects = {sum(sims_after)/len(sims_after):.4f}')
print(f'Individual similarities: {[f"{s:.3f}" for s in sims_after]}')
print(f'\nKey: MEMIT solves a constrained optimization for batch editing.')
print(f'Regularization prevents the update from being too large.')

## 3. SERAC与其他编辑方法

### SERAC (Scalable Efficient Model Editing with Counterfactual)
不修改原模型权重，而是训练一个反事实模型来处理编辑请求：
- 输入与编辑相关时，用反事实模型生成输出
- 输入与编辑无关时，用原模型生成输出
- 优势：不修改原模型，可逆

### 其他方法
| 方法 | 策略 | 可逆 | 批量编辑 | 精确度 |
|------|------|------|---------|--------|
| ROME | 权重修改 | 否 | 否 | 高 |
| MEMIT | 权重修改 | 否 | 是 | 高 |
| SERAC | 反事实模型 | 是 | 是 | 中 |
| CALINET | 注意力修改 | 否 | 否 | 中 |
| PMET | 提示编辑 | 是 | 是 | 中 |
| IKE | 上下文编辑 | 是 | 是 | 低 |

### 知识编辑评估
- **Efficacy**：编辑是否成功改变了目标知识
- **Generalization**：改写查询是否也被修改
- **Specificity**：无关知识是否被保持
- **Fluency**：生成质量是否下降

In [ ]:
class SERACEditor(nn.Module):
    def __init__(self, d_model=64, n_edits_max=10):
        super().__init__()
        self.d_model = d_model
        self.scope_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1), nn.Sigmoid())
        self.counterfactual = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, d_model))
        self.edits = []
        self.edit_embeds = nn.ParameterList()

    def add_edit(self, subject_embed, new_object_embed):
        self.edits.append({'subject': subject_embed.detach().clone(), 'object': new_object_embed.detach().clone()})
        self.edit_embeds.append(nn.Parameter(subject_embed.detach().clone()))

    def forward(self, query_embed, original_model_output):
        if not self.edits:
            return original_model_output
        max_relevance = 0
        most_relevant = None
        for i, edit in enumerate(self.edits):
            sim = F.cosine_similarity(query_embed.flatten().unsqueeze(0),
                                      edit['subject'].flatten().unsqueeze(0)).item()
            if sim > max_relevance:
                max_relevance = sim
                most_relevant = edit
        in_scope = self.scope_encoder(query_embed).item()
        if in_scope > 0.5 and most_relevant is not None:
            cf_output = self.counterfactual(query_embed)
            return in_scope * cf_output + (1 - in_scope) * original_model_output
        return original_model_output

class KnowledgeEditEvaluator:
    def __init__(self):
        self.results = []

    def evaluate_edit(self, model, editor, subject, new_object, unrelated_inputs):
        with torch.no_grad():
            edited_output = editor(subject, model(subject))
            efficacy = F.cosine_similarity(edited_output.flatten().unsqueeze(0),
                                            new_object.flatten().unsqueeze(0)).item()
            specificity_scores = []
            for unrelated in unrelated_inputs:
                original = model(unrelated)
                after_edit = editor(unrelated, original)
                spec = F.cosine_similarity(original.flatten().unsqueeze(0),
                                            after_edit.flatten().unsqueeze(0)).item()
                specificity_scores.append(spec)
            specificity = sum(specificity_scores) / len(specificity_scores) if specificity_scores else 1.0
        result = {'efficacy': efficacy, 'specificity': specificity}
        self.results.append(result)
        return result

serac = SERACEditor(d_model=64)
subject = torch.randn(1, 64)
new_obj = torch.randn(1, 64)
serac.add_edit(subject, new_obj)

original_output = torch.randn(1, 64)
edited_output = serac(subject, original_output)
unrelated_output = serac(torch.randn(1, 64), original_output)

print('=== SERAC: Counterfactual Model Editing ===')
print(f'Edited output change: {(edited_output - original_output).norm():.4f}')
print(f'Unrelated output change: {(unrelated_output - original_output).norm():.4f}')
print(f'\nKey: SERAC uses a counterfactual model for in-scope queries.')
print(f'Original model is untouched — edits are reversible.')
print(f'Scope encoder determines when to use the counterfactual model.')

## 4. 知识编辑总结

| 方法 | 修改对象 | 批量 | 可逆 | 精确度 | 产业适用 |
|------|---------|------|------|--------|----------|
| ROME | MLP权重 | 否 | 否 | 高 | 单条修正 |
| MEMIT | MLP权重 | 是 | 否 | 高 | 批量更新 |
| SERAC | 反事实模型 | 是 | 是 | 中 | 安全过滤 |
| IKE | 上下文提示 | 是 | 是 | 低 | 快速原型 |
| PMET | 提示参数 | 是 | 是 | 中 | 轻量编辑 |

**产业应用场景**：
- 事实更新：CEO更换、城市人口变化
- 错误修正：模型输出的错误知识
- 安全过滤：移除有害知识
- 合规要求：GDPR被遗忘权

**前沿方向**：
- **AlphaEdit**：基于投影的编辑
- **Eva-KELLM**：知识编辑评估基准
- **超网络编辑**：用超网络生成编辑权重